# 🔔 Clase 4 — Distribución normal
## Distribuciones de probabilidad y muestreo

**Desafío inicial:** En una empresa de manufactura se monitorean los tiempos de producción de una pieza clave. La duración sigue una distribución normal con **μ = 40 minutos** y **σ = 5 minutos**. El equipo necesita estimar la probabilidad de que una pieza tarde **más de 50 minutos**, para activar alertas operacionales automáticas.

**Objetivos:**
- Identificar las **características y propiedades** de la distribución normal
- Aplicar la **función de densidad** con `scipy.stats.norm`
- Calcular probabilidades usando la **distribución normal estándar**
- Aplicar la **transformación Z** para estandarizar variables
- Usar `norm.cdf()` y `norm.ppf()` correctamente

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, shapiro, kstest
import pandas as pd

plt.rcParams['figure.dpi'] = 110
plt.style.use('seaborn-v0_8-whitegrid')

print('✅ Librerías cargadas')

---
## PARTE 1 — Características de la distribución normal (slides 5-8)

In [ ]:
# Tabla de características — slide 6
tabla_car = pd.DataFrame({
    'Característica':    ['Forma','Simetría','Media = Mediana = Moda',
                          'Parámetros','Área total','Colas'],
    'Descripción': [
        'Campana de Gauss (forma de campana)',
        'Simétrica respecto a la media',
        'Los tres coinciden en el centro',
        'μ (media): centro | σ (desv.std): dispersión',
        'Área bajo la curva = 1 (100%)',
        'Asintóticas al eje x (nunca llegan a 0)'
    ]
})
print('=== Características de la distribución normal (slide 6) ===')
print(tabla_car.to_string(index=False))

In [ ]:
# Efecto de μ y σ sobre la forma de la distribución
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Parámetros μ y σ de la distribución normal', fontweight='bold')

x_plot = np.linspace(-5, 55, 500)

# Efecto de μ (misma σ)
for mu_val, col in [(10,'#FC4E4E'),(20,'#FFC000'),(30,'#70AD47'),(40,'#2E75B6')]:
    axes[0].plot(x_plot, norm.pdf(x_plot, loc=mu_val, scale=5),
                 color=col, linewidth=2, label=f'μ={mu_val}, σ=5')
axes[0].set_title('Efecto de μ (σ=5 constante)\nDesplaza la curva')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].legend(fontsize=8)

# Efecto de σ (misma μ)
for sigma_val, col in [(2,'#FC4E4E'),(4,'#FFC000'),(7,'#70AD47'),(10,'#2E75B6')]:
    y_s = norm.pdf(x_plot, loc=25, scale=sigma_val)
    axes[1].plot(x_plot, y_s, color=col, linewidth=2, label=f'σ={sigma_val}')
axes[1].set_title('Efecto de σ (μ=25 constante)\nControla el ancho')
axes[1].set_xlabel('x')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## PARTE 2 — Función de densidad (PDF) — código exacto slide 10

In [ ]:
# Código exacto de la presentación — slide 10
from scipy.stats import norm
import matplotlib.pyplot as plt
import numpy as np

mu, sigma = 40, 5
x = np.linspace(25, 55, 100)
y = norm.pdf(x, mu, sigma)

plt.plot(x, y)
plt.title('Distribución Normal con μ=40 y σ=5')
plt.xlabel('Tiempo de producción (min)')
plt.ylabel('Densidad de probabilidad')
plt.grid(True)
plt.show()

In [ ]:
# Visualización extendida: regla empírica + áreas clave
mu, sigma = 40, 5
x_ext = np.linspace(20, 60, 500)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x_ext, norm.pdf(x_ext, mu, sigma), color='#1F4E79', linewidth=2.5)

# Bandas de σ
zonas = [
    (mu-sigma, mu+sigma, '#70AD47', 0.4, f'μ±1σ [{mu-sigma},{mu+sigma}] ≈ 68%'),
    (mu-2*sigma, mu-sigma, '#FFC000', 0.3, ''),
    (mu+sigma, mu+2*sigma, '#FFC000', 0.3, f'μ±2σ [{mu-2*sigma},{mu+2*sigma}] ≈ 95%'),
    (mu-3*sigma, mu-2*sigma, '#ED7D31', 0.25, ''),
    (mu+2*sigma, mu+3*sigma, '#ED7D31', 0.25, f'μ±3σ [{mu-3*sigma},{mu+3*sigma}] ≈ 99.7%'),
]
for a_z, b_z, col, alp, lbl in zonas:
    xf = np.linspace(a_z, b_z, 100)
    ax.fill_between(xf, norm.pdf(xf, mu, sigma), alpha=alp, color=col,
                    label=lbl if lbl else None)

ax.axvline(mu, color='red', linestyle='--', linewidth=2, label=f'μ = {mu}')
for s in [-sigma, sigma, -2*sigma, 2*sigma]:
    ax.axvline(mu+s, color='gray', linestyle=':', linewidth=1, alpha=0.5)

ax.set_title(f'Distribución Normal (μ={mu}, σ={sigma}) — Regla empírica',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Tiempo de producción (min)')
ax.set_ylabel('Densidad de probabilidad')
ax.legend(fontsize=8)

# Anotación del desafío
p_gt50 = 1 - norm.cdf(50, mu, sigma)
ax.fill_between(np.linspace(50, 60, 100),
                norm.pdf(np.linspace(50, 60, 100), mu, sigma),
                alpha=0.7, color='#7030A0',
                label=f'P(X>50) = {p_gt50:.4f} ← DESAFÍO')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'\n🎯 RESPUESTA AL DESAFÍO:')
print(f'P(X > 50 min) = {p_gt50:.4f}  ({p_gt50*100:.2f}%)')
print(f'Aproximadamente el {p_gt50*100:.1f}% de las piezas tarda más de 50 min.')

---
## PARTE 3 — Transformación Z y cálculo de probabilidades (slides 12-14)

### 3.1 Código exacto slide 13

In [ ]:
# Código exacto de la presentación — slide 13
from scipy.stats import norm

prob = norm.cdf(47, loc=40, scale=5)
print(prob)   # Salida: ~0.9192

In [ ]:
# Transformación Z paso a paso — slide 12-13
mu, sigma = 40, 5
x_val = 47

Z = (x_val - mu) / sigma
prob_acum = norm.cdf(Z)   # equivalente a norm.cdf(47, 40, 5)

print('=== Transformación Z — slide 13 ===')
print(f'X ~ N(μ={mu}, σ={sigma}),  x = {x_val}')
print(f'Z = (x - μ) / σ = ({x_val} - {mu}) / {sigma} = {Z:.4f}')
print(f'P(X ≤ {x_val}) = P(Z ≤ {Z:.2f}) = {prob_acum:.4f}')
print(f'Interpretación: hay {prob_acum*100:.1f}% de probabilidad de que el')
print(f'tiempo de producción sea ≤ {x_val} minutos.')

In [ ]:
# Tipos de cálculos con norm.cdf() y norm.ppf() — slide 14
mu, sigma = 40, 5

print('=== Tipos de probabilidades calculables (slide 14) ===')
print()

# Cola izquierda: P(X ≤ x)
p_lt47 = norm.cdf(47, mu, sigma)
print(f'P(X ≤ 47) = norm.cdf(47, {mu}, {sigma}) = {p_lt47:.4f}')

# Cola derecha: P(X > x)
p_gt50 = 1 - norm.cdf(50, mu, sigma)
print(f'P(X > 50) = 1 - norm.cdf(50, {mu}, {sigma}) = {p_gt50:.4f}  ← DESAFÍO')

# Intervalo: P(a ≤ X ≤ b)
p_35_45 = norm.cdf(45, mu, sigma) - norm.cdf(35, mu, sigma)
print(f'P(35 ≤ X ≤ 45) = norm.cdf(45) - norm.cdf(35) = {p_35_45:.4f}')

# Percentil: valor correspondiente al percentil p
perc_90 = norm.ppf(0.90, mu, sigma)
print(f'Percentil 90 (ppf):    norm.ppf(0.90, {mu}, {sigma}) = {perc_90:.4f} min')
perc_95 = norm.ppf(0.95, mu, sigma)
print(f'Percentil 95 (ppf):    norm.ppf(0.95, {mu}, {sigma}) = {perc_95:.4f} min')
print()
print('Regla: cdf() → prob acumulada dado x | ppf() → x dado prob acumulada')

In [ ]:
# Visualización de todos los tipos de cálculo
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Tipos de cálculo de probabilidades con la normal (μ=40, σ=5)',
             fontweight='bold')
mu, sigma = 40, 5
x_plot = np.linspace(25, 55, 500)
y_plot = norm.pdf(x_plot, mu, sigma)

# Cola izquierda
axes[0].plot(x_plot, y_plot, color='#1F4E79', linewidth=2)
xf = np.linspace(25, 47, 200)
axes[0].fill_between(xf, norm.pdf(xf, mu, sigma), alpha=0.4, color='#2E75B6')
axes[0].axvline(47, color='red', linestyle='--', linewidth=2)
axes[0].set_title(f'P(X ≤ 47) = {p_lt47:.4f}\n(cola izquierda)')
axes[0].set_xlabel('min')

# Cola derecha
axes[1].plot(x_plot, y_plot, color='#1F4E79', linewidth=2)
xf2 = np.linspace(50, 55, 200)
axes[1].fill_between(xf2, norm.pdf(xf2, mu, sigma), alpha=0.4, color='#FC4E4E')
axes[1].axvline(50, color='red', linestyle='--', linewidth=2)
axes[1].set_title(f'P(X > 50) = {p_gt50:.4f}\n(cola derecha — desafío)')
axes[1].set_xlabel('min')

# Intervalo
axes[2].plot(x_plot, y_plot, color='#1F4E79', linewidth=2)
xf3 = np.linspace(35, 45, 200)
axes[2].fill_between(xf3, norm.pdf(xf3, mu, sigma), alpha=0.4, color='#70AD47')
axes[2].axvline(35, color='red', linestyle='--', linewidth=2)
axes[2].axvline(45, color='red', linestyle='--', linewidth=2)
axes[2].set_title(f'P(35 ≤ X ≤ 45) = {p_35_45:.4f}\n(intervalo)')
axes[2].set_xlabel('min')

plt.tight_layout()
plt.show()

### 3.2 Verificación de normalidad — buenas prácticas slide 8

In [ ]:
# Herramientas para verificar normalidad
import statsmodels.api as sm

np.random.seed(42)
# Simular datos de producción
datos_prod = norm.rvs(loc=40, scale=5, size=200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Verificación de normalidad — Herramientas (slide 8)', fontweight='bold')

# 1. Histograma
axes[0].hist(datos_prod, bins=20, density=True, alpha=0.7, color='#2E75B6', edgecolor='white')
x_norm = np.linspace(25, 55, 200)
axes[0].plot(x_norm, norm.pdf(x_norm, datos_prod.mean(), datos_prod.std()),
             color='red', linewidth=2, label='Normal ajustada')
axes[0].set_title('1. Histograma + curva normal')
axes[0].set_xlabel('Tiempo (min)')
axes[0].legend(fontsize=8)

# 2. Q-Q Plot
sm.qqplot(datos_prod, line='s', ax=axes[1])
axes[1].set_title('2. Q-Q Plot\n(puntos ≈ línea → normalidad)')

# 3. Tests estadísticos
stat_sw, p_sw = shapiro(datos_prod[:50])   # Shapiro-Wilk (n<50)
stat_ks, p_ks = kstest(datos_prod, 'norm',
                        args=(datos_prod.mean(), datos_prod.std()))

axes[2].axis('off')
texto = [
    '3. Tests estadísticos',
    '',
    f'Shapiro-Wilk (n=50):',
    f'  W = {stat_sw:.4f}',
    f'  p = {p_sw:.4f}',
    f'  {"✅ Normal (p>0.05)" if p_sw>0.05 else "⚠️ No normal (p<0.05)"}',
    '',
    f'Kolmogorov-Smirnov (n=200):',
    f'  D = {stat_ks:.4f}',
    f'  p = {p_ks:.4f}',
    f'  {"✅ Normal (p>0.05)" if p_ks>0.05 else "⚠️ No normal (p<0.05)"}',
]
axes[2].text(0.05, 0.95, '\n'.join(texto), transform=axes[2].transAxes,
             fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='#F0F8FF', alpha=0.8))

plt.tight_layout()
plt.show()

---
## PARTE 4 — Actividad guiada: Empresa de logística (slides 15-18)

### 4.1 Código exacto slide 18

In [ ]:
# Código exacto de la presentación — slide 18
from scipy.stats import norm

# Ejemplo: calcular probabilidad fuera del rango 14-22
prob = norm.cdf(14, loc=18, scale=2.5) + (1 - norm.cdf(22, loc=18, scale=2.5))
print(f"Probabilidad fuera del rango: {prob:.4f}")

In [ ]:
# Análisis completo — instrucciones slides 16-17
mu_log, sigma_log = 18, 2.5

print('=== PASO 2: Transformación Z ===')
for x_val, desc in [(14,'límite inferior'),(22,'límite superior'),(18,'media'),(23,'percentil 97')]:
    z = (x_val - mu_log) / sigma_log
    print(f'  x={x_val} ({desc}): Z = ({x_val}-{mu_log})/{sigma_log} = {z:.4f}')

print()
print('=== PASO 3: Probabilidades ===')

p_lt14 = norm.cdf(14, mu_log, sigma_log)
p_gt22 = 1 - norm.cdf(22, mu_log, sigma_log)
p_fuera = p_lt14 + p_gt22
p_dentro = 1 - p_fuera
p_sla    = norm.cdf(22, mu_log, sigma_log) - norm.cdf(14, mu_log, sigma_log)

print(f'P(X < 14h) — muy rápida:        {p_lt14:.4f}  ({p_lt14*100:.1f}%)')
print(f'P(X > 22h) — tarde:              {p_gt22:.4f}  ({p_gt22*100:.1f}%)')
print(f'P(fuera del rango 14-22h):       {p_fuera:.4f}  ({p_fuera*100:.1f}%)')
print(f'P(dentro del rango 14-22h) SLA:  {p_sla:.4f}  ({p_sla*100:.1f}%)')
print()

# Percentiles operacionales
perc_95 = norm.ppf(0.95, mu_log, sigma_log)
perc_99 = norm.ppf(0.99, mu_log, sigma_log)
print(f'Percentil 95 (alerta): {perc_95:.1f}h  → 5% de entregas supera esto')
print(f'Percentil 99 (crítico): {perc_99:.1f}h  → 1% de entregas supera esto')

In [ ]:
# Visualización de la actividad guiada
x_plot = np.linspace(10, 26, 500)
y_plot = norm.pdf(x_plot, mu_log, sigma_log)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Actividad guiada — Tiempos de entrega express (μ={mu_log}h, σ={sigma_log}h)',
             fontweight='bold', fontsize=12)

# PDF con zonas coloreadas
axes[0].plot(x_plot, y_plot, color='#1F4E79', linewidth=2.5)

# Zona de alerta baja (< 14h)
xf1 = np.linspace(10, 14, 100)
axes[0].fill_between(xf1, norm.pdf(xf1, mu_log, sigma_log),
                      alpha=0.5, color='#FFC000',
                      label=f'P(X<14h) = {p_lt14:.3f}')

# Zona normal (SLA)
xf2 = np.linspace(14, 22, 300)
axes[0].fill_between(xf2, norm.pdf(xf2, mu_log, sigma_log),
                      alpha=0.3, color='#70AD47',
                      label=f'SLA: P(14≤X≤22) = {p_sla:.3f}')

# Zona de alerta alta (> 22h)
xf3 = np.linspace(22, 26, 100)
axes[0].fill_between(xf3, norm.pdf(xf3, mu_log, sigma_log),
                      alpha=0.5, color='#FC4E4E',
                      label=f'P(X>22h) = {p_gt22:.3f}')

axes[0].axvline(mu_log, color='navy', linestyle='--', linewidth=2, label=f'μ={mu_log}h')
axes[0].axvline(14, color='orange', linestyle=':', linewidth=2)
axes[0].axvline(22, color='red', linestyle=':', linewidth=2)
axes[0].set_title('Zonas de alerta operativa')
axes[0].set_xlabel('Horas')
axes[0].set_ylabel('Densidad')
axes[0].legend(fontsize=8)

# CDF con percentiles clave
axes[1].plot(x_plot, norm.cdf(x_plot, mu_log, sigma_log),
             color='#2E75B6', linewidth=2.5)
for perc, pval, col in [
    (14, p_lt14, '#FFC000'),
    (22, norm.cdf(22, mu_log, sigma_log), '#FC4E4E'),
    (perc_95, 0.95, '#7030A0'),
]:
    axes[1].plot([10, perc], [pval, pval], color=col, linestyle='--', linewidth=1.5)
    axes[1].plot([perc, perc], [0, pval],   color=col, linestyle='--', linewidth=1.5)
    axes[1].scatter([perc],[pval], color=col, s=80, zorder=5)
    axes[1].text(perc+0.1, pval-0.04, f'{perc:.1f}h\n({pval*100:.1f}%)',
                 fontsize=8, color=col)

axes[1].set_title('CDF — F(x) = P(X ≤ x)')
axes[1].set_xlabel('Horas')
axes[1].set_ylabel('Probabilidad acumulada')
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

### 4.2 Plantilla de resolución — instrucciones slide 17

In [ ]:
# Plantilla editable — pasos 1-5 del slide 17

# PASO 1: Identificar parámetros
mu_p,  sigma_p  = 18, 2.5
lim_inf, lim_sup = 14, 22

# PASO 2: Transformar a valores Z
Z_inf = (lim_inf - mu_p) / sigma_p
Z_sup = (lim_sup - mu_p) / sigma_p

# PASO 3: Calcular probabilidades
p_inf = norm.cdf(lim_inf, mu_p, sigma_p)
p_sup = 1 - norm.cdf(lim_sup, mu_p, sigma_p)
p_total_fuera = p_inf + p_sup

# PASO 4: Registrar interpretación
interpretacion = ""

# PASO 5: Implicancias operativas
implicancias = ""

print('=' * 60)
print('  PLANTILLA — Probabilidades bajo la distribución normal')
print('=' * 60)
print(f'  Variable: X ~ N(μ={mu_p}, σ={sigma_p})')
print(f'  Rango aceptable: [{lim_inf}h, {lim_sup}h]')
print()
print(f'  PASO 2 — Transformación Z:')
print(f'    Z_inf = ({lim_inf}-{mu_p})/{sigma_p} = {Z_inf:.4f}')
print(f'    Z_sup = ({lim_sup}-{mu_p})/{sigma_p} = {Z_sup:.4f}')
print()
print(f'  PASO 3 — Probabilidades:')
print(f'    P(X < {lim_inf}h) = {p_inf:.4f}  ({p_inf*100:.2f}%)')
print(f'    P(X > {lim_sup}h) = {p_sup:.4f}  ({p_sup*100:.2f}%)')
print(f'    P(fuera del rango) = {p_total_fuera:.4f}  ({p_total_fuera*100:.2f}%)')
print()
print(f'  PASO 4 — Interpretación: {interpretacion}')
print(f'  PASO 5 — Implicancias:   {implicancias}')
print('=' * 60)

In [ ]:
# Casos adicionales del Casos_distribuciones.csv — slide 18
try:
    df_casos = pd.read_csv('Casos_distribuciones.csv')
    print('=== Casos disponibles para práctica ===')
    print(df_casos[['Caso','Fenomeno','Distribucion_sugerida','Pregunta_probabilidad']].to_string(index=False))
except FileNotFoundError:
    print('Archivo Casos_distribuciones.csv no encontrado.')
    print('Genera el archivo con el código de la Clase 2 o descárgalo desde los materiales.')

---
## PARTE 5 — Aplicación práctica: datos de manufactura simulados

In [ ]:
# Simular dataset de producción y verificar normalidad
np.random.seed(42)
n_piezas = 500
tiempos  = norm.rvs(loc=40, scale=5, size=n_piezas)

mu_obs   = tiempos.mean()
sigma_obs = tiempos.std(ddof=1)

print(f'=== Estadísticos de la muestra simulada (n={n_piezas}) ===')
print(f'  Media muestral:    {mu_obs:.4f}  (teórica: 40)')
print(f'  Desv.std muestral: {sigma_obs:.4f}  (teórica: 5)')
print()

# Probabilidades empíricas vs teóricas
p_gt50_teo = 1 - norm.cdf(50, 40, 5)
p_gt50_emp = (tiempos > 50).mean()
p_lt30_teo = norm.cdf(30, 40, 5)
p_lt30_emp = (tiempos < 30).mean()

print('=== Comparación teórica vs empírica ===')
print(f'P(X > 50): Teórica={p_gt50_teo:.4f} | Empírica={p_gt50_emp:.4f}')
print(f'P(X < 30): Teórica={p_lt30_teo:.4f} | Empírica={p_lt30_emp:.4f}')

### ✏️ Preguntas de cierre — slide 21:

In [ ]:
p1 = ""  # Características fundamentales de la normal
p2 = ""  # Cómo interpretar la función de densidad
p3 = ""  # Qué representa la transformación Z
p4 = ""  # Cómo calcular probabilidad entre dos valores
p5 = ""  # Cuándo es inapropiado aplicar la normal

preguntas = [
    '¿Cuáles son las características fundamentales de una distribución normal?',
    '¿Cómo se interpreta la función de densidad de una distribución normal?',
    '¿Qué representa la transformación Z y cómo se aplica?',
    '¿Cómo se calcula la probabilidad de que una variable continua esté entre dos valores?',
    '¿Cuándo es inapropiado aplicar la distribución normal en un problema real?'
]

print('--- PREGUNTAS DE CIERRE (slide 21) ---')
for i, (preg, resp) in enumerate(zip(preguntas, [p1,p2,p3,p4,p5]), 1):
    print(f'\n{i}. {preg}')
    print(f'   R: {resp}')

---
## 📋 Resumen

### Propiedades de la distribución normal

| Propiedad | Descripción |
|-----------|-------------|
| Forma | Campana simétrica |
| Centro | μ = Media = Mediana = Moda |
| Dispersión | σ (desviación estándar) |
| Área total | 1.0 (100%) |
| Regla 68-95-99.7 | P(\|X-μ\| ≤ kσ) ≈ 68/95/99.7% para k=1,2,3 |

### Transformación Z

$$Z = \frac{X - \mu}{\sigma}$$

Convierte cualquier X~N(μ,σ) a Z~N(0,1)

### Funciones clave de `scipy.stats.norm`

| Función | Descripción | Ejemplo |
|---------|-------------|--------|
| `norm.pdf(x, μ, σ)` | Densidad en el punto x | `norm.pdf(47, 40, 5)` |
| `norm.cdf(x, μ, σ)` | P(X ≤ x) acumulada | `norm.cdf(47, 40, 5)` |
| `1 - norm.cdf(x, μ, σ)` | P(X > x) | `1 - norm.cdf(50, 40, 5)` |
| `norm.cdf(b) - norm.cdf(a)` | P(a ≤ X ≤ b) | `norm.cdf(45)-norm.cdf(35)` |
| `norm.ppf(p, μ, σ)` | Percentil p → valor x | `norm.ppf(0.95, 40, 5)` |

> 💡 **`cdf` vs `pdf`:** `pdf(x)` da la densidad (no es probabilidad). Para obtener probabilidades siempre usa `cdf`. La probabilidad puntual P(X=x) = 0 en distribuciones continuas.

> 💡 **Verificar antes de aplicar:** usa histograma + Q-Q plot + Shapiro-Wilk para confirmar normalidad. No asumas normalidad sin evidencia.